[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/24_rope.ipynb)

# 🔴 Hard: Rotary Position Embedding (RoPE)

Implement **RoPE** — the position encoding used in LLaMA, GPT-NeoX, and most modern LLMs.

### Signature
```python
def apply_rope(q: Tensor, k: Tensor) -> tuple[Tensor, Tensor]:
    # q, k: (B, S, D) where D is even
    # Returns rotated (q, k) with same shape
```

### Key Idea
Split each vector into consecutive pairs. Rotate each pair by `θ = pos / 10000^(2i/D)`:
```
[x_0, x_1] → [x_0*cosθ - x_1*sinθ, x_0*sinθ + x_1*cosθ]
```
This makes `dot(q_rot[i], k_rot[j])` depend only on `i - j` (relative position).

In [9]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.1 MB/s eta 0:00:00


In [10]:
import torch
import math

In [11]:
# ✏️ YOUR IMPLEMENTATION HERE
def rotate(x: torch.tensor) -> torch.tensor:
  B, S, D = x.shape
  x = x.view(B, S, D//2, 2)

  x0 = x[..., 0] # [B, S, D//2]
  x1 = x[..., 1] # [B, S, D//2]

  assert x0.shape == (B, S, D//2), f"{x0.shape}"

  pos = torch.arange(0, S).unsqueeze(-1) #[S, 1]
  i = torch.arange(0, D, 2).unsqueeze(0) # [1, D//2]
  theta = pos / torch.pow(10000, i/D) # [S, D//2]
  assert theta.shape == (S, D//2)

  x0_after = x0 * torch.cos(theta) - x1 * torch.sin(theta)
  x1_after = x0 * torch.sin(theta) + x1 * torch.cos(theta)

  res = torch.stack((x0, x1), dim=-1) # [B, S, D//2, 2]
  return res.view(B, S, D)


def apply_rope(q, k):
    # 1. Compute position angles
    # 2. Split into even/odd pairs
    # 3. Apply rotation
    return rotate(q), rotate(k)

In [12]:
# 🧪 Debug
q = torch.randn(1, 8, 16)
k = torch.randn(1, 8, 16)
qr, kr = apply_rope(q, k)
print('Shape preserved:', qr.shape == q.shape)
print('Norm preserved:', torch.allclose(q.norm(dim=-1), qr.norm(dim=-1), atol=1e-4))

Shape preserved: True
Norm preserved: True


In [13]:
# ✅ SUBMIT
from torch_judge import check
check('rope')


🧪 Testing: Rotary Position Embedding (RoPE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shapes (6.6ms)
  ✅ [2/4] Preserves norm (8.6ms)
  ✅ [3/4] Relative position property (30.1ms)
  ✅ [4/4] Gradient flow (36.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (81.5ms total)
  Progress saved. Run status() to see your dashboard.

